# Fase 1 · Paso 3: cierre de la revisión bilateral

Esta libreta valida el registro visual ciego ya completado, congela los parámetros aceptados y genera la evidencia de cierre. No procesa la cohorte completa, no recorta articulaciones, no entrena y no abre la prueba reservada.

## 1. Montar Drive y localizar la evidencia privada

In [ ]:
import os
from pathlib import Path
from google.colab import drive, userdata

drive.mount('/content/drive')
private_root = userdata.get('KNEE_DATA_ROOT') or os.environ.get('KNEE_DATA_ROOT')
if not private_root:
    raise RuntimeError('Configura el secreto KNEE_DATA_ROOT antes de continuar.')
os.environ['KNEE_DATA_ROOT'] = private_root
private_root_path = Path(private_root)
review_path = private_root_path / 'outputs' / 'auditorias' / 'fase_1' / 'paso_3_separacion_bilateral' / 'revision_visual_ciega.csv'
if not review_path.is_file():
    raise FileNotFoundError('No se encontró el registro de revisión visual ciega.')
print('Drive montado y registro privado de revisión localizado.')

## 2. Obtener una revisión identificable del repositorio

In [ ]:
import subprocess

REPO_URL = 'https://github.com/AlonsoYess/Knee.git'
BRANCH = 'codex/fase-0-base-reproducible'
REPO_DIR = Path('/content/Knee_fase1_paso3_cierre')

if REPO_DIR.exists():
    if not (REPO_DIR / '.git').is_dir():
        raise RuntimeError('La ruta de trabajo existe y no es un clon Git; usa un entorno nuevo.')
    dirty = subprocess.run(['git', 'status', '--porcelain'], cwd=REPO_DIR, capture_output=True, text=True, check=True).stdout
    if dirty:
        raise RuntimeError('El clon temporal contiene cambios. Restablece el entorno de ejecución.')
    subprocess.run(['git', 'fetch', 'origin', BRANCH], cwd=REPO_DIR, check=True)
    subprocess.run(['git', 'checkout', BRANCH], cwd=REPO_DIR, check=True)
    subprocess.run(['git', 'merge', '--ff-only', f'origin/{BRANCH}'], cwd=REPO_DIR, check=True)
else:
    subprocess.run(['git', 'clone', '--branch', BRANCH, '--single-branch', REPO_URL, str(REPO_DIR)], check=True)
commit = subprocess.run(['git', 'rev-parse', 'HEAD'], cwd=REPO_DIR, capture_output=True, text=True, check=True).stdout.strip()
print('Revisión preparada:', commit)

## 3. Instalar y comprobar el código

In [ ]:
import sys

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO_DIR)], check=True)
subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', 'tests', '-v'], cwd=REPO_DIR, check=True)
if subprocess.run(['git', 'status', '--porcelain'], cwd=REPO_DIR, capture_output=True, text=True, check=True).stdout:
    raise RuntimeError('Las comprobaciones modificaron el repositorio temporal.')
print('Código y pruebas automáticas preparados.')

## 4. Validar la revisión y congelar los parámetros

In [ ]:
config_path = REPO_DIR / 'configs' / 'bilateral_review.example.json'
subprocess.run([sys.executable, '-m', 'knee.bilateral_review', '--config', str(config_path), '--git-commit', commit], cwd=REPO_DIR, check=True)

## 5. Registrar el cierre controlado

In [ ]:
import json
from datetime import datetime, timezone

output_dir = private_root_path / 'outputs' / 'auditorias' / 'fase_1' / 'paso_3_separacion_bilateral'
closure_path = output_dir / 'cierre_paso_3_publico.json'
closure = json.loads(closure_path.read_text(encoding='utf-8'))
if closure.get('status') != 'closed' or not closure.get('parameters_frozen'):
    raise RuntimeError('El cierre del paso 3 no quedó validado.')
record = {
    'phase': 1,
    'step': 3,
    'operation': 'close_blinded_bilateral_review',
    'executed_at_utc': datetime.now(timezone.utc).isoformat(),
    'git_commit': commit,
    'status': closure['status'],
    'algorithm_version': closure['algorithm_version'],
    'reviewed_unique_studies': closure['reviewed_unique_studies'],
    'split_accepted': closure['split_accepted'],
    'laterality_mapping_supported': closure['laterality_mapping_supported'],
    'technical_exclusions': closure['technical_exclusions'],
    'parameters_frozen': closure['parameters_frozen'],
    'mass_processing_executed': False,
    'training_executed': False,
    'reserved_test_opened': False,
}
(output_dir / 'registro_cierre.json').write_text(json.dumps(record, indent=2, ensure_ascii=False) + '\n', encoding='utf-8')
print('Fase 1, paso 3 cerrado técnicamente. Parámetros congelados; procesamiento masivo y entrenamiento continúan bloqueados.', record)